In [1]:
!pip install streamlit ultralytics transformers faiss-cpu ranx
!npm install -g localtunnel

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 87.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 74.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 114.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 467.3/467.3 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 81.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.6 MB/s eta 0:00:00
  Created wheel for wa

In [9]:
%%writefile app.py
import streamlit as st
import torch, os, faiss, json, numpy as np, pandas as pd
from PIL import Image
from transformers import CLIPProcessor, CLIPModel, Blip2Processor, Blip2ForConditionalGeneration
from transformers import AutoImageProcessor, AutoModelForObjectDetection
import torch.nn.functional as F
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN, add_to_git_credential=False)
except Exception:
    HF_TOKEN = None

st.set_page_config(page_title="Visual Fashion Search", page_icon="S", layout="wide", initial_sidebar_state="collapsed")

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@400;700;900&family=DM+Sans:wght@300;400;500&display=swap');
:root{--bg-primary:#0a0a0a;--bg-secondary:#111111;--bg-card:#161616;--accent:#c8a96e;--accent-light:#e2c99a;--text-primary:#f0ede8;--text-muted:#8a8580;--border:#2a2a2a;--success:#4caf87;--error:#e05c5c}
*{font-family:'DM Sans',sans-serif}
.stApp{background:var(--bg-primary)!important;color:var(--text-primary)!important}
#MainMenu,footer,header{visibility:hidden}.stDeployButton{display:none}
.hero{text-align:center;padding:3rem 0 2rem;border-bottom:1px solid var(--border);margin-bottom:2.5rem}
.hero-logo{font-family:'Playfair Display',serif;font-size:4rem;font-weight:900;letter-spacing:-0.02em;color:var(--text-primary);line-height:1;margin-bottom:0.5rem}
.hero-logo span{color:var(--accent)}
.hero-tagline{font-size:0.95rem;color:var(--text-muted);letter-spacing:0.3em;text-transform:uppercase;font-weight:300}
.step-label{display:inline-flex;align-items:center;gap:0.75rem;margin-bottom:1.25rem}
.step-number{width:2rem;height:2rem;border-radius:50%;background:var(--accent);color:#000;font-weight:700;font-size:0.85rem;display:flex;align-items:center;justify-content:center}
.step-title{font-family:'Playfair Display',serif;font-size:1.4rem;font-weight:700;color:var(--text-primary)}
.result-rank{font-family:'Playfair Display',serif;font-size:1.5rem;font-weight:900;color:var(--accent)}
.result-score{display:inline-block;background:rgba(200,169,110,0.15);color:var(--accent-light);border:1px solid rgba(200,169,110,0.3);border-radius:20px;padding:0.2rem 0.75rem;font-size:0.8rem;font-weight:500;margin:0.4rem 0}
.result-item-id{font-size:0.7rem;color:var(--text-muted);font-family:monospace;margin:0.3rem 0}
.result-caption{font-size:0.78rem;color:var(--text-muted);font-style:italic;line-height:1.4;margin-top:0.4rem}
.divider{border:none;border-top:1px solid var(--border);margin:2rem 0}
.badge-success{display:inline-flex;align-items:center;gap:0.4rem;background:rgba(76,175,135,0.12);color:var(--success);border:1px solid rgba(76,175,135,0.3);border-radius:20px;padding:0.3rem 0.9rem;font-size:0.82rem;font-weight:500}
.badge-warning{display:inline-flex;align-items:center;gap:0.4rem;background:rgba(224,92,92,0.12);color:var(--error);border:1px solid rgba(224,92,92,0.3);border-radius:20px;padding:0.3rem 0.9rem;font-size:0.82rem;font-weight:500}
.query-label{font-family:'Playfair Display',serif;color:var(--text-muted);margin-bottom:0.5rem;letter-spacing:0.1em;text-transform:uppercase;font-size:0.75rem}
.stButton>button{background:transparent!important;border:1px solid var(--border)!important;color:var(--text-primary)!important;border-radius:8px!important;font-weight:500!important;padding:0.6rem 1.5rem!important;transition:all 0.2s!important;width:100%}
.stButton>button:hover{border-color:var(--accent)!important;color:var(--accent)!important}
.stButton>button[kind="primary"]{background:var(--accent)!important;border-color:var(--accent)!important;color:#000!important;font-weight:600!important}
.stButton>button[kind="primary"]:hover{background:var(--accent-light)!important;border-color:var(--accent-light)!important;color:#000!important}
div[data-testid="stFileUploadDropzone"]{background:var(--bg-card)!important;border:2px dashed var(--border)!important;border-radius:12px!important;padding:2rem!important}
div[data-testid="stFileUploadDropzone"]:hover{border-color:var(--accent)!important}
.stSpinner>div{border-top-color:var(--accent)!important}
.stTable table{background:var(--bg-card)!important;color:var(--text-primary)!important;border:1px solid var(--border)!important;border-radius:8px!important}
.stTable th{background:var(--bg-secondary)!important;color:var(--accent)!important;font-weight:600!important;font-size:0.8rem!important;text-transform:uppercase!important}
.stTable td{color:var(--text-primary)!important;border-color:var(--border)!important;font-size:0.85rem!important}
.loading-bar{width:100%;height:2px;background:var(--border);border-radius:2px;overflow:hidden;margin:1rem 0}
.loading-bar-fill{height:100%;background:linear-gradient(90deg,var(--accent),var(--accent-light),var(--accent));background-size:200% 100%;animation:shimmer 1.5s infinite;border-radius:2px}
@keyframes shimmer{0%{background-position:200% 0}100%{background-position:-200% 0}}
.model-status{display:flex;gap:0.5rem;flex-wrap:wrap;margin-top:1rem}
.model-pill{background:rgba(76,175,135,0.1);border:1px solid rgba(76,175,135,0.25);color:var(--success);border-radius:20px;padding:0.25rem 0.75rem;font-size:0.75rem;font-weight:500}
</style>
""", unsafe_allow_html=True)

FASHION_DETECTOR  = 'valentinafeve/yolos-fashionpedia'
CLIP_MODEL_PATH   = '/kaggle/input/datasets/hades199/3c-clip-fintuned-model/clip_finetuned_v2/seed_623/best'
FAISS_INDEX_PATH  = '/kaggle/input/datasets/hades199/3c-faiss-indexes/faiss_indexes/C_finetuned_seed623_alpha0.7_index.bin'
FAISS_META_PATH   = '/kaggle/input/datasets/hades199/3c-faiss-indexes/faiss_indexes/C_finetuned_seed623_alpha0.7_metadata.json'
GALLERY_IMG_DIR   = '/kaggle/input/datasets/hades199/yolo-cropped-images'
RAW_IMG_DIR       = '/kaggle/input/datasets/hades199/vr-final-project-dataset/vr-final-project-dataset'
CAPTION_FILE      = '/kaggle/input/datasets/hades199/3c-blip-captions/captions.json'
TOP_K             = 10
RERANK_TOP_N      = 50
DEVICE            = "cuda" if torch.cuda.is_available() else "cpu"
DETECT_THRESHOLD  = 0.4

# Only detect main garment categories (skip parts like sleeve, pocket, zipper)
MAIN_GARMENTS = {
    'shirt, blouse', 'top, t-shirt, sweatshirt', 'sweater', 'cardigan',
    'jacket', 'vest', 'pants', 'shorts', 'skirt', 'coat', 'dress',
    'jumpsuit', 'cape', 'tie', 'hat', 'glasses', 'glove', 'belt',
    'scarf', 'shoe', 'bag, wallet', 'tights, stockings', 'sock',
}

# Map each garment to Upper/Lower/Full for FAISS metadata filtering
GARMENT_TO_REGION = {
    'shirt, blouse': 'Upper', 'top, t-shirt, sweatshirt': 'Upper',
    'sweater': 'Upper', 'cardigan': 'Upper', 'jacket': 'Upper',
    'vest': 'Upper', 'coat': 'Upper', 'cape': 'Upper',
    'tie': 'Upper', 'scarf': 'Upper', 'glasses': 'Upper', 'hat': 'Upper',
    'glove': 'Upper',
    'pants': 'Lower', 'shorts': 'Lower', 'skirt': 'Lower',
    'tights, stockings': 'Lower', 'sock': 'Lower', 'shoe': 'Lower',
    'belt': 'Lower',
    'dress': 'Full', 'jumpsuit': 'Full', 'bag, wallet': 'Full',
}

# ── Load Models ──
@st.cache_resource
def load_fashion_detector():
    proc  = AutoImageProcessor.from_pretrained(FASHION_DETECTOR)
    model = AutoModelForObjectDetection.from_pretrained(FASHION_DETECTOR).to(DEVICE)
    model.eval()
    return model, proc

@st.cache_resource
def load_clip():
    model = CLIPModel.from_pretrained(CLIP_MODEL_PATH).to(DEVICE)
    processor = CLIPProcessor.from_pretrained(CLIP_MODEL_PATH)
    model.eval()
    return model, processor

@st.cache_resource
def load_blip2():
    kwargs = {"token": HF_TOKEN} if HF_TOKEN else {}
    processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b", use_fast=False, **kwargs)
    model = Blip2ForConditionalGeneration.from_pretrained("Salesforce/blip2-opt-2.7b", torch_dtype=torch.float16, device_map="auto", **kwargs)
    model.eval()
    return model, processor

@st.cache_resource
def load_faiss():
    index = faiss.read_index(FAISS_INDEX_PATH)
    hnsw = faiss.downcast_index(index.index)
    hnsw.hnsw.efSearch = 128
    with open(FAISS_META_PATH, 'r') as f:
        metadata = json.load(f)
    return index, metadata

@st.cache_resource
def load_captions():
    with open(CAPTION_FILE, 'r') as f:
        return json.load(f)

# ── Pipeline Functions ──
def detect_clothing_items(image, det_model, det_processor):
    """Detect specific clothing items using YOLOS-Fashionpedia.
    Returns one best detection per garment category, sorted by confidence."""
    inputs = det_processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = det_model(**inputs)

    target_sizes = torch.tensor([image.size[::-1]]).to(DEVICE)
    results = det_processor.post_process_object_detection(
        outputs, target_sizes=target_sizes, threshold=DETECT_THRESHOLD
    )[0]

    all_dets = []
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        label_name = det_model.config.id2label[label.item()]
        if label_name not in MAIN_GARMENTS:
            continue  # skip parts like sleeve, pocket, zipper
        conf = score.item()
        x1, y1, x2, y2 = map(int, box.tolist())
        # Clamp to image bounds
        w, h = image.size
        x1, y1, x2, y2 = max(0,x1), max(0,y1), min(w,x2), min(h,y2)
        if x2 - x1 < 10 or y2 - y1 < 10:
            continue  # skip tiny detections
        crop = image.crop((x1, y1, x2, y2))
        meta_class = GARMENT_TO_REGION.get(label_name, 'Upper')
        display_name = label_name.replace(', ', ' / ').title()
        all_dets.append({
            "class_name": display_name,
            "meta_class": meta_class,
            "confidence": conf,
            "bbox": (x1, y1, x2, y2),
            "crop": crop,
        })

    # Keep only the best detection per garment category
    all_dets.sort(key=lambda d: d["confidence"], reverse=True)
    seen = set()
    detections = []
    for d in all_dets:
        if d["class_name"] not in seen:
            seen.add(d["class_name"])
            detections.append(d)
    return detections

def get_clip_embedding(image, clip_model, clip_processor):
    inputs = clip_processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        vision_outputs = clip_model.vision_model(pixel_values=inputs["pixel_values"])
        image_embeds = clip_model.visual_projection(vision_outputs.pooler_output)
    return F.normalize(image_embeds, dim=-1).cpu().numpy().astype(np.float32)

def blip2_itm_score(query_img, candidate_captions, blip_model, blip_processor):
    scores = []
    BATCH = 8
    pad_token_id = blip_processor.tokenizer.pad_token_id
    for i in range(0, len(candidate_captions), BATCH):
        batch_caps = candidate_captions[i:i+BATCH]
        inputs = blip_processor(images=[query_img]*len(batch_caps), text=batch_caps, return_tensors="pt", padding=True, truncation=True, max_length=77).to(DEVICE, torch.float16)
        with torch.no_grad():
            outputs = blip_model(**inputs)
            logits = outputs.logits
        labels = inputs["input_ids"]
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
        token_losses = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1)).view(len(batch_caps), -1)
        valid_tokens = (shift_labels != pad_token_id).float()
        sequence_loss = (token_losses * valid_tokens).sum(dim=-1) / valid_tokens.sum(dim=-1)
        scores.extend((-sequence_loss).cpu().numpy().tolist())
    return scores

# ── Hero ──
st.markdown("""
<div class="hero">
    <div class="hero-logo">Visual<span>Search</span></div>
    <div class="hero-tagline">Visual Fashion Product Search Engine</div>
</div>
""", unsafe_allow_html=True)

# ── Load Models ──
if 'models_loaded' not in st.session_state:
    with st.spinner("Initialising models - this takes ~2 minutes on first load..."):
        st.markdown('<div class="loading-bar"><div class="loading-bar-fill"></div></div>', unsafe_allow_html=True)
        det_model, det_proc   = load_fashion_detector()
        clip_model, clip_proc = load_clip()
        blip_model, blip_proc = load_blip2()
        faiss_index, metadata = load_faiss()
        captions              = load_captions()
        st.session_state['models_loaded'] = True
else:
    det_model, det_proc   = load_fashion_detector()
    clip_model, clip_proc = load_clip()
    blip_model, blip_proc = load_blip2()
    faiss_index, metadata = load_faiss()
    captions              = load_captions()

st.markdown("""
<div class="model-status">
    <span class="model-pill">&#10003; YOLOS Fashion</span>
    <span class="model-pill">&#10003; CLIP (fine-tuned)</span>
    <span class="model-pill">&#10003; BLIP-2</span>
    <span class="model-pill">&#10003; FAISS HNSW</span>
    <span class="model-pill">&#10003; Captions</span>
</div>
""", unsafe_allow_html=True)

st.markdown('<hr class="divider">', unsafe_allow_html=True)

# ── K selector (main area) ──
k_options = [5, 10, 15, 20]
TOP_K = st.select_slider("Number of results (K)", options=k_options, value=10)

# ── Sidebar ──
with st.sidebar:
    st.markdown("### Settings")
    RERANK_TOP_N = st.slider("Candidates before reranking", 20, 100, 50)
    beta         = st.slider("BLIP-2 reranking weight", 0.0, 1.0, 0.3, 0.05, help="0 = pure FAISS | 1 = pure BLIP-2 ITM")
    filter_class = st.checkbox("Filter results by clothing type", value=True, help="Only show results matching the selected body region")
    st.markdown("---")
    st.markdown("### Pipeline")
    st.markdown("""
    1. **YOLOS** - detect clothing items
    2. **Select item** - choose what to search
    3. **CLIP** - visual embedding
    4. **FAISS HNSW** - ANN search
    5. **BLIP-2 ITM** - semantic reranking
    """)

# ── Step 1 - Upload ──
st.markdown("""
<div class="step-label">
    <div class="step-number">1</div>
    <div class="step-title">Upload a Product Image</div>
</div>
""", unsafe_allow_html=True)

uploaded = st.file_uploader("Drop a clothing image here", type=["jpg", "jpeg", "png"], label_visibility="collapsed")

if uploaded is not None:
    raw_img = Image.open(uploaded).convert("RGB")
    st.markdown('<hr class="divider">', unsafe_allow_html=True)

    # ── Step 2 - Fashion Detection ──
    st.markdown("""
    <div class="step-label">
        <div class="step-number">2</div>
        <div class="step-title">Clothing Item Detection</div>
    </div>
    """, unsafe_allow_html=True)

    with st.spinner("Detecting clothing items..."):
        detections = detect_clothing_items(raw_img, det_model, det_proc)

    col_orig, col_det = st.columns(2, gap="large")
    with col_orig:
        st.markdown('<div class="query-label">Original Image</div>', unsafe_allow_html=True)
        st.image(raw_img, use_container_width=True)
    with col_det:
        if detections:
            det_text = ", ".join([d['class_name'] for d in detections])
            st.markdown(f'<div class="query-label">Detected: <span class="badge-success">{len(detections)} item(s) — {det_text}</span></div>', unsafe_allow_html=True)
            for d in detections:
                st.markdown(f"- **{d['class_name']}** — {d['confidence']:.0%} confidence → *{d['meta_class']} body*")
        else:
            st.markdown('<div class="query-label"><span class="badge-warning">No items detected — will use full image</span></div>', unsafe_allow_html=True)

    st.markdown('<hr class="divider">', unsafe_allow_html=True)

    # ── Step 3 - Select Item ──
    st.markdown("""
    <div class="step-label">
        <div class="step-number">3</div>
        <div class="step-title">Select Clothing Item</div>
    </div>
    <p style="color:#8a8580; font-size:0.9rem; margin-bottom:1rem;">
        Choose which detected item you want to search for in the catalog.
    </p>
    """, unsafe_allow_html=True)

    # Fallback: if nothing detected, offer full image
    if not detections:
        detections = [{
            "class_name": "Full Image",
            "meta_class": "Full",
            "crop": raw_img,
            "confidence": 0.0,
            "bbox": (0, 0, raw_img.size[0], raw_img.size[1]),
        }]

    # Show detected item crops dynamically
    n_items = len(detections)
    cols = st.columns(min(n_items, 5), gap="small")
    for i, det in enumerate(detections):
        with cols[i % min(n_items, 5)]:
            st.image(det['crop'], use_container_width=True)
            st.caption(f"{det['class_name']} ({det['confidence']:.0%})")

    radio_labels = [det['class_name'] for det in detections]
    selected_item = st.radio("Search for:", radio_labels, index=0, horizontal=True)
    sel_pos = radio_labels.index(selected_item)

    col_confirm, _ = st.columns([2, 3])
    with col_confirm:
        confirm = st.button("Search with Selected Item", type="primary", use_container_width=True)

    if confirm:
        chosen = detections[sel_pos]
        st.session_state['query_img']   = chosen['crop']
        st.session_state['query_class']  = chosen['meta_class']
        st.session_state['query_label']  = chosen['class_name']
        st.session_state['confirmed']    = True
        st.session_state['results']      = None

    # ── Step 4 - Search ──
    if st.session_state.get('confirmed', False):
        query_img   = st.session_state['query_img']
        query_label = st.session_state.get('query_label', 'Selected')
        query_class = st.session_state.get('query_class', None)

        st.markdown('<hr class="divider">', unsafe_allow_html=True)
        class_info = f" | Filtering: {query_class}" if (filter_class and query_class) else ""
        st.markdown(f"""
        <div class="step-label">
            <div class="step-number">4</div>
            <div class="step-title">Searching Catalog</div>
        </div>
        <p style="color:#8a8580; font-size:0.9rem; margin-bottom:1rem;">
            Using {query_label.lower()} | CLIP &rarr; FAISS HNSW &rarr; BLIP-2 reranking{class_info}
        </p>
        """, unsafe_allow_html=True)

        if st.session_state.get('results') is None:
            with st.spinner("Computing CLIP embedding..."):
                query_embed = get_clip_embedding(query_img, clip_model, clip_proc)

            with st.spinner("Searching gallery..."):
                search_k = RERANK_TOP_N * 3 if (filter_class and query_class) else RERANK_TOP_N
                D, I = faiss_index.search(query_embed, k=search_k)

                filter_relaxed = False  # track if we had to drop the class filter

                def build_candidates(indices, distances, apply_filter):
                    cands = []
                    for fid, fscore in zip(indices, distances):
                        meta = metadata.get(str(fid))
                        if not meta:
                            continue
                        if apply_filter and query_class and meta.get('clothes_type'):
                            if meta['clothes_type'] != query_class:
                                continue
                        image_name = meta.get('image_name', '')
                        item_id    = meta.get('item_id', '')
                        caption = captions.get(image_name, captions.get(image_name.replace('/', '_'), "a clothing item"))
                        raw_path  = os.path.join(RAW_IMG_DIR, image_name)
                        crop_path = os.path.join(GALLERY_IMG_DIR, image_name)
                        if not os.path.exists(crop_path):
                            crop_path = os.path.join(GALLERY_IMG_DIR, image_name.replace('/', '_'))
                        display_path = raw_path if os.path.exists(raw_path) else (crop_path if os.path.exists(crop_path) else None)
                        # Cosine similarity from squared-L2 distance (valid for unit-norm vectors)
                        cosine_sim = max(0.0, 1.0 - fscore / 2.0)
                        cands.append({
                            'faiss_id': fid, 'faiss_score': fscore,
                            'cosine_sim': cosine_sim,
                            'image_name': image_name, 'item_id': item_id,
                            'caption': caption, 'clothes_type': meta.get('clothes_type', ''),
                            'img_path': display_path
                        })
                    return cands[:RERANK_TOP_N]

                candidates = build_candidates(I[0].tolist(), D[0].tolist(), apply_filter=filter_class)

                # ── Fallback: if class filtering wiped all candidates, retry without filter ──
                if len(candidates) == 0 and filter_class and query_class:
                    filter_relaxed = True
                    # Fetch a fresh wider search without the class constraint
                    D2, I2 = faiss_index.search(query_embed, k=RERANK_TOP_N * 2)
                    candidates = build_candidates(I2[0].tolist(), D2[0].tolist(), apply_filter=False)

                if filter_relaxed:
                    st.warning(
                        f"⚠️ No **{query_class}** body items found in the catalog for this query. "
                        f"Showing the best visual matches across all clothing types instead."
                    )

            with st.spinner("BLIP-2 semantic reranking..."):
                if candidates:
                    itm_scores = blip2_itm_score(query_img, [c['caption'] for c in candidates], blip_model, blip_proc)
                    faiss_arr = np.array([c['faiss_score'] for c in candidates])
                    itm_arr   = np.array(itm_scores)

                    def norm01(arr):
                        mn, mx = arr.min(), arr.max()
                        return np.ones_like(arr) if mx - mn < 1e-8 else (arr - mn) / (mx - mn)

                    faiss_norm = 1.0 - norm01(faiss_arr)
                    itm_norm   = norm01(itm_arr)
                    combined   = (1 - beta) * faiss_norm + beta * itm_norm

                    for i, (itm, final) in enumerate(zip(itm_scores, combined)):
                        candidates[i]['itm_score']   = itm
                        candidates[i]['final_score'] = float(final)

                    results = sorted(candidates, key=lambda x: x['final_score'], reverse=True)[:TOP_K]
                else:
                    results = []

            st.session_state['results'] = results

        results = st.session_state['results']

        # ── Step 5 - Results ──
        st.markdown('<hr class="divider">', unsafe_allow_html=True)
        st.markdown(f"""
        <div class="step-label">
            <div class="step-number">5</div>
            <div class="step-title">Top {TOP_K} Similar Products</div>
        </div>
        """, unsafe_allow_html=True)

        if not results:
            st.error("No results found. Try a different image or disable class filtering in the sidebar.")
        else:
            q_col, res_col = st.columns([1, 4], gap="large")
            with q_col:
                st.markdown('<div class="query-label">Your Query</div>', unsafe_allow_html=True)
                st.image(query_img, use_container_width=True)
                st.markdown(f'<div class="result-score" style="margin-top:0.5rem">K={TOP_K} | {query_label}</div>', unsafe_allow_html=True)

            with res_col:
                COLS_PER_ROW = 5
                for row_start in range(0, len(results), COLS_PER_ROW):
                    row_results = results[row_start:row_start+COLS_PER_ROW]
                    cols = st.columns(COLS_PER_ROW, gap="small")
                    for col_idx, (col, result) in enumerate(zip(cols, row_results)):
                        with col:
                            rank = row_start + col_idx + 1
                            if result['img_path'] and os.path.exists(result['img_path']):
                                try:
                                    st.image(Image.open(result['img_path']).convert("RGB"), use_container_width=True)
                                except Exception:
                                    st.markdown("[image]")
                            else:
                                st.markdown("[image]")
                            cap_short = result['caption'][:60] + "..." if len(result['caption']) > 60 else result['caption']
                            ctype = f" | {result.get('clothes_type','')}" if result.get('clothes_type') else ""
                            # cosine_sim: CLIP visual similarity (pre-rerank)
                            clip_sim_pct = result['cosine_sim'] * 100
                            # final_score: combined CLIP + BLIP-2 rerank score [0,1]
                            final_pct = result['final_score'] * 100
                            st.markdown(f"""
<div style="text-align:center; padding:0.25rem 0">
    <div class="result-rank">#{rank}</div>
    <div class="result-score" title="Combined CLIP + BLIP-2 rerank score">{final_pct:.1f}% match{ctype}</div>
    <div style="font-size:0.72rem;color:#8a8580;margin:0.15rem 0" title="Raw CLIP cosine similarity">CLIP sim: {clip_sim_pct:.1f}%</div>
    <div class="result-item-id">{result['item_id']}</div>
    <div class="result-caption">{cap_short}</div>
</div>
""", unsafe_allow_html=True)

            st.markdown('<hr class="divider">', unsafe_allow_html=True)
            st.markdown('<div style="font-family:Playfair Display,serif;font-size:1.1rem;color:#f0ede8;margin-bottom:1rem">Results Summary</div>', unsafe_allow_html=True)
            summary_df = pd.DataFrame([{
                'Rank': i+1,
                'Item ID': r['item_id'],
                'Type': r.get('clothes_type',''),
                'CLIP Cosine Sim': f"{r['cosine_sim']*100:.1f}%",
                'Combined Score': f"{r['final_score']:.4f}",
                'BLIP-2 ITM': f"{r.get('itm_score', 0.0):.4f}",
                'Caption': r['caption'][:70]+'...' if len(r['caption'])>70 else r['caption']
            } for i, r in enumerate(results)])
            st.table(summary_df)

        st.markdown('<hr class="divider">', unsafe_allow_html=True)
        col_reset, _ = st.columns([1, 3])
        with col_reset:
            if st.button("New Search", use_container_width=True):
                for key in ['confirmed', 'query_img', 'results', 'query_class', 'query_label']:
                    st.session_state[key] = None
                st.rerun()

else:
    st.markdown("""
    <div style="text-align:center; padding: 4rem 2rem; color:#4a4744; border: 1px dashed #2a2a2a; border-radius:16px; margin-top:1rem;">
        <div style="font-size:3rem; margin-bottom:1rem;">&#128087;</div>
        <div style="font-family:'Playfair Display',serif; font-size:1.5rem; color:#6a6764; margin-bottom:0.5rem;">
            Drop an image to begin
        </div>
        <div style="font-size:0.85rem; color:#4a4744;">
            Upload any clothing product image — JPG or PNG<br>
            <strong style="color:#8a8580;">NEW:</strong> Detects individual clothing items (jacket, pants, dress, etc.)
        </div>
    </div>
    """, unsafe_allow_html=True)


Overwriting app.py


In [3]:
!wget -q -O - ipv4.icanhazip.com

34.53.79.167


In [4]:
!pip install streamlit pyngrok -q

In [5]:
!pkill -f streamlit
!pkill -f ngrok

In [6]:
from pyngrok import ngrok
ngrok.set_auth_token("3CO2HKJlOp7tBvU11yx5Mr3Ljid_MCbc6qX9qDrBhW2tUm5h")

In [10]:
import subprocess
import time
from pyngrok import ngrok

# 1. Kill any existing tunnels
ngrok.kill()
# 3. Open a text file to capture the background logs
log_file = open("server_logs.txt", "w")

print("Starting Streamlit on fresh Port 8503... (Logs are being recorded!)")

# 4. Start Streamlit on 8503
process = subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8503",            
    "--server.address", "0.0.0.0",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
], stdout=log_file, stderr=subprocess.STDOUT)

time.sleep(5)

# 5. Connect Ngrok to 8503
tunnel = ngrok.connect(8503)            # <-- FRESH PORT
print(f"\n✅ App is live at: {tunnel.public_url}")
print("⏳ Keeping server alive... (Do not stop this cell)")

try:
    process.wait()
except KeyboardInterrupt:
    print("\nShutting down server...")
    ngrok.kill()
    process.kill()
    log_file.close()

Starting Streamlit on fresh Port 8503... (Logs are being recorded!)

✅ App is live at: https://wolverine-extrude-outgoing.ngrok-free.dev
⏳ Keeping server alive... (Do not stop this cell)


t=2026-05-15T13:31:29+0000 lvl=warn msg="Stopping forwarder" name=http-8503-83bf0dd5-2651-44d1-bbb9-7f2912053d8c acceptErr="failed to accept connection: Listener closed"
t=2026-05-15T13:31:29+0000 lvl=warn msg="Error restarting forwarder" name=http-8503-83bf0dd5-2651-44d1-bbb9-7f2912053d8c err="failed to start tunnel: remote gone away"



Shutting down server...


In [ ]:
!cat server_logs.txt